In [1]:
%load_ext autoreload
%autoreload 

In [2]:
import os

os.environ['MUJOCO_GL'] = 'egl'

In [3]:
import sys

# Add parent directory to path for imports
sys.path.insert(0, "/home/hgf_hmgu/hgf_gib4562/tdmpc2/")
sys.path.insert(0, "/home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2/")

import matplotlib.pyplot as plt
import numpy as np
import pickle
from tqdm import tqdm
import pandas as pd
import json
from pathlib import Path
import torch

import numpy as np
import seaborn as sns
import os
#from analysis.utils import load_sweep_metadata, load_data, compute_success, chunk_data
from analysis.utils_decoding import evaluate_lags_leave_k_out

from hydra import initialize, compose
from omegaconf import OmegaConf
from common.parser import parse_cfg
from common.seed import set_seed

from tdmpc2 import TDMPC2

In [4]:
from envs import make_env

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card3: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card2: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card0: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card3: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card2: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card0: Permission denied



## Initialize TDMPC2 agents

In [5]:
# Base configuration for state-based environment
CKPT_TO_USE = 500_000
CKPT_PATH_STATE = f'logs/model-runs/2025-12-20/19-10-12/cartpole_exp/models/{CKPT_TO_USE}.pt'
CKPT_PATH_RGB = '/home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2/outputs/2026-01-15/21-56-52/cartpole_exp_pixel/models/650000.pt'

override_cfg_state = dict(
    task='cartpole-swingup',
    checkpoint=CKPT_PATH_STATE,
    obs='state',
    seed=0,
    compile=False,
    mpc=True,
    multitask=False,
    model_size=5,
    save_video=False,
    record_planning=False,
)

override_cfg_rgb = dict(
    task='cartpole-swingup',
    checkpoint=CKPT_PATH_RGB,
    obs='rgb',
    seed=0,
    compile=False,
    mpc=True,
    multitask=False,
    model_size=5,
    save_video=False,
    record_planning=False,
)

# Create RGB-based environment
with initialize(config_path="tdmpc2", version_base=None):
    cfg_rgb = compose(config_name="config")
    OmegaConf.set_struct(cfg_rgb, False)
    cfg_rgb = OmegaConf.merge(cfg_rgb, override_cfg_rgb)

cfg_rgb = parse_cfg(cfg_rgb)
set_seed(cfg_rgb.seed)

cfg_rgb.initial_state = {
    'qpos': {
        'slider': 0,
        'hinge_1': 0,
    },
}

env_rgb = make_env(cfg_rgb)

print("\nLoading RGB-based agent...")
agent_rgb = TDMPC2(cfg_rgb)
if hasattr(cfg_rgb, 'checkpoint') and cfg_rgb.checkpoint:
    print(f"Loading checkpoint: {cfg_rgb.checkpoint}")
    agent_rgb.load(cfg_rgb.checkpoint)

# Create state-based environment
with initialize(config_path="tdmpc2", version_base=None):
    cfg_state = compose(config_name="config")
    OmegaConf.set_struct(cfg_state, False)
    cfg_state = OmegaConf.merge(cfg_state, override_cfg_state)

cfg_state = parse_cfg(cfg_state)
set_seed(cfg_state.seed)

cfg_state.initial_state = {
    'qpos': {
        'slider': 0,
        'hinge_1': 0,
    },
}

env_state = make_env(cfg_state)

print("\nLoading state-based agent...")
agent_state = TDMPC2(cfg_state)
if hasattr(cfg_state, 'checkpoint') and cfg_state.checkpoint:
    print(f"Loading checkpoint: {cfg_state.checkpoint}")
    agent_state.load(cfg_state.checkpoint)



Loading RGB-based agent...
Episode length: 500
Discount factor: 0.99
Loading checkpoint: /home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2/outputs/2026-01-15/21-56-52/cartpole_exp_pixel/models/650000.pt

Loading state-based agent...
Episode length: 500
Discount factor: 0.99
Loading checkpoint: logs/model-runs/2025-12-20/19-10-12/cartpole_exp/models/500000.pt


In [6]:
import pickle

with open(
        "/home/hgf_hmgu/hgf_gib4562/tdmpc2/analysis/data_generation/data_off_policy.pkl",
        "rb") as f:
    episodes = pickle.load(f)

In [10]:
obs_episodes_state = episodes['obs_states']
obs_episodes_pixel = episodes['obs_pixels_tdmpc2']
actions_episodes = episodes['actions']
rewards_episodes = episodes['rewards']

obs_episodes_state.shape, obs_episodes_pixel.shape, actions_episodes.shape, rewards_episodes.shape

((25000, 5), (25000, 9, 64, 64), (25000, 1), (25000, 1))

## Convert observations to activations

In [12]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

sys.path.insert(0, '/home/hgf_hmgu/hgf_gib4562/tdmpc2/tdmpc2')
from tdmpc2 import TDMPC2
from tqdm import tqdm
from envs import make_env
from hydra import initialize, compose
from omegaconf import OmegaConf
from common.parser import parse_cfg
from common.seed import set_seed
from analysis.utils import chunk_data

## tdmpc2 activations

In [11]:
agent_state.eval()
agent_rgb.eval()
assert not agent_state.model._encoder.training
assert not agent_rgb.model._encoder.training

In [13]:
zs_tdmpc2_state = []
for ep in chunk_data(obs_episodes_state, 500):
    with torch.no_grad():
        x = torch.from_numpy(ep).cuda()
        z = agent_state.model.encode(x, task=None).detach().cpu().numpy()
        zs_tdmpc2_state.append(z)

zs_tdmpc2_state_all = np.concatenate(zs_tdmpc2_state, axis=0)

zs_tdmpc2_pixel = []
for ep in chunk_data(obs_episodes_pixel, 500):
    with torch.no_grad():
        x = torch.from_numpy(ep).cuda()
        z = agent_rgb.model.encode(x, task=None).detach().cpu().numpy()
        zs_tdmpc2_pixel.append(z)

zs_tdmpc2_pixel_all = np.concatenate(zs_tdmpc2_pixel, axis=0)

In [14]:
zs_tdmpc2_pixel_all.shape, zs_tdmpc2_state_all.shape

((25000, 512), (25000, 512))

## Decoding

In [19]:
vars_all = {
    'z_tdmpc2_state': zs_tdmpc2_state_all,
    'z_tdmpc2_pixel': zs_tdmpc2_pixel_all,
    'position': obs_episodes_state[:, 0].reshape(-1, 1),
    'cos(pole_angle)': obs_episodes_state[:, 1].reshape(-1, 1),
    'sin(pole_angle)': obs_episodes_state[:, 2].reshape(-1, 1),
    'cart_velocity': obs_episodes_state[:, 3].reshape(-1, 1),
    'pole_angular_velocity': obs_episodes_state[:, 4].reshape(-1, 1),
    'rewards': rewards_episodes,
    'actions': actions_episodes,
}

for k, v in vars_all.items():
    print(k, v.shape)


z_tdmpc2_state (25000, 512)
z_tdmpc2_pixel (25000, 512)
position (25000, 1)
cos(pole_angle) (25000, 1)
sin(pole_angle) (25000, 1)
cart_velocity (25000, 1)
pole_angular_velocity (25000, 1)
rewards (25000, 1)
actions (25000, 1)


In [23]:
decoding_relationships = [
    ('position', 'position'),
    ('cos(pole_angle)', 'cos(pole_angle)'),
    ('sin(pole_angle)', 'sin(pole_angle)'),
    ('cart_velocity', 'cart_velocity'),
    ('pole_angular_velocity', 'pole_angular_velocity'),
    ('rewards', 'rewards'),
    ('actions', 'actions'),
    ('z_tdmpc2_pixel', 'position'),
    ('z_tdmpc2_pixel', 'cos(pole_angle)'),
    ('z_tdmpc2_pixel', 'sin(pole_angle)'),
    ('z_tdmpc2_pixel', 'cart_velocity'),
    ('z_tdmpc2_pixel', 'pole_angular_velocity'),
    ('z_tdmpc2_pixel', 'rewards'),
    ('z_tdmpc2_pixel', 'actions'),
    ('z_tdmpc2_state', 'position'),
    ('z_tdmpc2_state', 'cos(pole_angle)'),
    ('z_tdmpc2_state', 'sin(pole_angle)'),
    ('z_tdmpc2_state', 'cart_velocity'),
    ('z_tdmpc2_state', 'pole_angular_velocity'),
    ('z_tdmpc2_state', 'rewards'),
    ('z_tdmpc2_state', 'actions'),
]


In [24]:
path_to_save = '/home/hgf_hmgu/hgf_gib4562/tdmpc2/analysis/results_decoding'

results_decoding = {}
for x_var, y_var in decoding_relationships:
    print(f"Decoding {x_var} -> {y_var}")
    results_decoding[f'{x_var}-{y_var}'] = evaluate_lags_leave_k_out(
        chunk_data(vars_all[x_var], 500),
        chunk_data(vars_all[y_var], 500),
        model='linear_regression',
        lags=np.arange(-30, 31, 5),
        k_holdout=1,
        add_bootstrap_ci=True,
        max_combinations=25,
        shuffle_X=False,
        shuffle_y=False,
        predict_difference=False,
        standardize_X=True
    )  #NOTE: I'm standardizing the X data, but not the y data

#Save results_decoding to pickle file
with open(
        os.path.join(
            path_to_save,
            f'results_decoding_linear_regression_offpolicy_tdmpc2_state_and_pixel.pkl'
        ), 'wb') as f:
    pickle.dump(results_decoding, f)

Decoding position -> position


100%|██████████| 25/25 [00:00<00:00, 34.46it/s]


{-30: 0.5486053621768952, -25: 0.6632325351238251, -20: 0.7691557312011719, -15: 0.8614656519889832, -10: 0.9346685624122619, -5: 0.9828575348854065, 0: 1.0, 5: 0.9824492621421814, 10: 0.9312992668151856, 15: 0.849786639213562, 20: 0.740732307434082, 25: 0.6061829900741578, 30: 0.44703556776046754}
Decoding cos(pole_angle) -> cos(pole_angle)


100%|██████████| 25/25 [00:00<00:00, 34.48it/s]


{-30: -0.5653601431846619, -25: -0.5794341475516558, -20: -0.5625208090990782, -15: -0.49800125606358053, -10: -0.1832066234946251, -5: 0.5427530598640442, 0: 1.0, 5: 0.5415969157218933, 10: -0.18749223828315734, 15: -0.4983257011324167, 20: -0.5476680672168732, 25: -0.548798105083406, 30: -0.5282359737716615}
Decoding sin(pole_angle) -> sin(pole_angle)


...

{-30: 0.7377649891376495, -25: 0.8136392295360565, -20: 0.8678776264190674, -15: 0.9039957523345947, -10: 0.9302343654632569, -5: 0.9647449684143067, 0: 0.9941023588180542, 5: 0.9780707716941833, 10: 0.9615011429786682, 15: 0.9351896619796753, 20: 0.8985422420501709, 25: 0.862913248538971, 30: 0.8038313353061676}
Decoding z_tdmpc2_state -> rewards


100%|██████████| 25/25 [02:46<00:00,  6.65s/it]


{-30: 0.06141262681242567, -25: 0.23291009918621416, -20: 0.5030146438330273, -15: 0.7446569958934431, -10: 0.882403638442035, -5: 0.9540836758407153, 0: 0.9783528868747049, 5: 0.973997608557009, 10: 0.9486375731902522, 15: 0.8892210545366173, 20: 0.7683483164816284, 25: 0.5736347046511017, 30: 0.3648367058290299}
Decoding z_tdmpc2_state -> actions


100%|██████████| 25/25 [02:46<00:00,  6.68s/it]


{-30: 0.3408968567103148, -25: 0.37664238452911375, -20: 0.39411714792251584, -15: 0.38183855533599853, -10: 0.32763098761439324, -5: 0.2214822231978178, 0: 0.062058733105659486, 5: -0.15442251309752464, 10: -0.3673672550916672, 15: -0.5536506736278534, 20: -0.6965215256810189, 25: -0.8012166924774646, 30: -0.8732544070482254}
